# 📈 Análise de Rentabilidade: Produtos e Canais

## Objetivo de Negócio
Atendendo às diretrizes da Diretoria, esta análise visa responder objetivamente:
1. **Quem são os 'Heróis da Margem'?** (Produtos que garantem a sustentabilidade da empresa).
2. **Quem são os 'Vilões'?** (Produtos que vendem muito, mas corroem o lucro por conta de altos custos ou descontos).

**Metodologia:** Utilizaremos a nossa `Fato de Vendas` (tratada, livre de vazamento temporal) para construir a **Curva ABC** de margem de lucro bruto.

In [1]:
import pandas as pd
import duckdb
import matplotlib.pyplot as plt
import seaborn as sns

# Configuração visual (estilo limpo executivo)
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

# Carregando a Base Fato
con = duckdb.connect()
df = con.execute("SELECT * FROM '../data/processed/fato_vendas.parquet'").df()
print(f"Dados carregados: {len(df):,} linhas de itens vendidos.")

Dados carregados: 134,417 linhas de itens vendidos.


### 1. Curva ABC de Produtos (Focada em Margem Bruta)
No varejo tradicional, muitos focam na *Receita*. Vamos focar no *Lucro Bruto* (`gross_margin`).

In [2]:
# Agrupando por Produto (SKU)
df_prod = df.groupby(['sku', 'product_name']).agg({
    'item_revenue': 'sum',
    'total_cost': 'sum',
    'gross_margin': 'sum',
    'quantity': 'sum'
}).reset_index()

# Calculando a Margem Bruta Percentual (%)
df_prod['margin_pct'] = (df_prod['gross_margin'] / df_prod['item_revenue']) * 100

# Ordenando pelos que mais geram DINHEIRO (Massa de Margem)
df_prod = df_prod.sort_values('gross_margin', ascending=False)

# Calculando o percentual acumulado (Regra 80/20)
df_prod['margin_cumsum_pct'] = df_prod['gross_margin'].cumsum() / df_prod['gross_margin'].sum() * 100

# Classificando A, B e C
def classify_abc(pct):
    if pct <= 80:
        return 'A'
    elif pct <= 95:
        return 'B'
    else:
        return 'C'

df_prod['curva'] = df_prod['margin_cumsum_pct'].apply(classify_abc)

# Visualizando o Top 10 Produtos - Nossos "Heróis"
display(df_prod.head(10).style.format({
    'item_revenue': 'R$ {:,.2f}',
    'gross_margin': 'R$ {:,.2f}',
    'margin_pct': '{:.1f}%'
}))

,sku,product_name,item_revenue,total_cost,gross_margin,quantity,margin_pct,margin_cumsum_pct,curva
871,LHN-866836,Cabo Náutico 5117,"R$ 3,551,646.42",1667438.640000,"R$ 1,884,207.78",882,53.1%,0.337488,A
587,LHN-588613,Defensa Náutica 2623,"R$ 3,397,401.61",1595023.380000,"R$ 1,802,378.23",823,53.1%,0.660320,A
330,LHN-353117,Colete Salva-Vidas 2374,"R$ 3,338,419.05",1552755.150000,"R$ 1,785,663.90",1005,53.5%,0.980158,A
411,LHN-424788,Colete Salva-Vidas 3398,"R$ 3,448,039.65",1665720.450000,"R$ 1,782,319.20",885,51.7%,1.299397,A
461,LHN-477634,Bússola de Bordo 1248,"R$ 3,377,037.96",1639337.290000,"R$ 1,737,700.67",827,51.5%,1.610644,A
895,LHN-886714,Cabo Náutico 5921,"R$ 3,310,560.13",1614908.240000,"R$ 1,695,651.89",881,51.2%,1.914359,A
71,LHN-067210,Bateria Náutica 5523,"R$ 3,108,420.42",1425879.420000,"R$ 1,682,541.00",714,54.1%,2.215726,A
754,LHN-758750,Sonar Transducer 2387,"R$ 3,122,393.40",1486855.440000,"R$ 1,635,537.96",756,52.4%,2.508674,A
903,LHN-900424,Tinta Antifouling 8939,"R$ 3,141,598.98",1517682.140000,"R$ 1,623,916.84",782,51.7%,2.799541,A
899,LHN-894118,Hélice de Alumínio 8577,"R$ 2,999,495.70",1382255.100000,"R$ 1,617,240.60",790,53.9%,3.089212,A


### 2. Identificando os "Vilões" da Rentabilidade
Queremos produtos da Curva A em Receita (vendem muito em volume), mas que deixam pouquíssima Margem (< 5%) ou até Margem Negativa.

In [3]:
# Filtrando produtos com alta receita, mas margem baixa
high_revenue_threshold = df_prod['item_revenue'].quantile(0.80) # Top 20% em vendas brutas
viloes = df_prod[(df_prod['item_revenue'] >= high_revenue_threshold) & (df_prod['margin_pct'] < 10)].sort_values('margin_pct')

if len(viloes) > 0:
    print(f"Foram encontrados {len(viloes)} produtos com alto faturamento e margem preocupante (<10%).")
    display(viloes.head(10))
else:
    print("Não temos 'vilões' críticos na margem base (sem considerar devoluções)!")

Não temos 'vilões' críticos na margem base (sem considerar devoluções)!


### 3. Exemplo Prático do Framework de Decisão
*(Preencha de acordo com os resultados das células acima)*

**Fato Observado:** 
*Não temos 'vilões' críticos na margem base (sem considerar devoluções)! E nossos top 10 produtos possuem margin_pct > 50%*

**Hipótese:**
*A ausências de devoluções pode influenciar nesse resultado, considerar uma análise mais aprofundada*

**Recomendação:**
*No momento sem insights práticos.*